In [1]:
import matplotlib.pyplot as plt
import numpy as np
from keras.models import Sequential
from keras.layers import Input, Dense
from keras.datasets import california_housing

(inputs, targets), (x_test, y_test) = california_housing.load_data(version='large', 
                                        test_split=0.2)
print(inputs.shape, targets.shape)
print(x_test.shape, y_test.shape)

(16512, 8) (16512,)
(4128, 8) (4128,)


In [6]:
# tarkastellaan jakoa
print(f"treenidatan prosentuaalinen koko: {inputs.shape[0] * 100 / (inputs.shape[0] + x_test.shape[0])}%")

print(f"testidatan prosentuaalinen koko: {x_test.shape[0] * 100 / (inputs.shape[0] + x_test.shape[0])}%")

treenidatan prosentuaalinen koko: 80.0%
testidatan prosentuaalinen koko: 20.0%


## Tarkastelaan mikä on noin 20% validointidataa varte

In [8]:
print(inputs.shape[0]*0.2)

3302.4


In [10]:
# Datan jako treeni ja validointi dataan

x_train = inputs[:3302]
y_train = targets[:3302]
x_val = inputs[3302:]
y_val = targets[3302:]

## Standardoidaan treenidata

In [14]:
# Keskiarvo ja keskihajonta lasketaan vain treenidatasta
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
x_train -= mean
x_train /= std
x_test -= mean
x_test /= std
x_val -= mean
x_val /= std

# Tarkastellaan 5 ensimmäistä tulosta
print(f"x_train: {x_train[:5]}\n")
print(f"\nx_test: {x_test[:5]}\n")
print(f"\n*x_val:{x_val[:5]}")

x_train: [[ 0.6468956  -0.7262112   1.846102   -0.1418168   0.0246513  -0.32271007
  -0.03514035 -0.02148999]
 [ 0.60196555 -0.7872938  -0.6166024  -0.37251562  0.02691753 -0.12917632
   0.12836729 -0.5960081 ]
 [-1.4098116   0.9981811   1.846102   -0.33955863 -0.26542667 -0.59593415
  -0.26553747  0.08049066]
 [ 0.80663955 -0.8060887  -1.4110233   0.20972426  0.3668526   0.1212791
   0.46776956  0.28195   ]
 [-1.4747062   1.073359   -0.3782762  -0.9793634  -1.0880696  -1.0423628
  -1.0657341   5.851833  ]]


x_test: [[ 6.0196555e-01 -7.3090899e-01  1.2900074e+00 -1.9982107e-01
  -1.3851753e-01 -1.4625281e-01 -1.2432635e-01 -2.3812401e-01]
 [-3.1656146e-01 -4.7248533e-01 -1.1726971e+00 -1.2003692e-03
  -6.5998055e-02  3.4270234e-02 -4.2572521e-02  3.4139797e-01]
 [-8.2075489e-01  1.0216757e+00 -1.6493496e+00 -3.3692208e-01
  -5.5777079e-01 -3.6255524e-01 -5.0584424e-01  7.4447972e-01]
 [-1.2450745e+00  1.0404688e+00 -9.3437076e-01  2.1203499e+00
   2.2931514e+00  5.2623624e-01  2.33819

In [17]:
# Tarkastellaan kohdearvoja (10 ensimmäistä)
print(f"y_train (10 ensimmäistä):\n{y_train[:10]}\n")
print(f"y_test (10 ensimmäistä):\n{y_test[:10]}\n")
print(f"y_val (10 ensimmäistä):\n{y_val[:10]}")

y_train (10 ensimmäistä):
[252300. 146900. 290900. 141300. 500001.  72000. 252400. 162500. 500001.
 226800.]

y_test (10 ensimmäistä):
[397900. 227900. 172100. 186500. 148900.  65600. 344600. 235600. 233000.
 192100.]

y_val (10 ensimmäistä):
[228900.  71900. 272200. 395300. 316700. 177500. 500000. 267500. 105100.
 339700.]


In [19]:
# arvot ovat 100 tuhansissa tai lähellä -> jaetaan 100000:lla
y_train /= 1e5
y_test /= 1e5
y_val /= 1e5

# Tarkastellaan kohdearvoja jaon jälkeen (10 ensimmäistä)
print(f"y_train (10 ensimmäistä):\n{y_train[:10]}\n")
print(f"y_test (10 ensimmäistä):\n{y_test[:10]}\n")
print(f"y_val (10 ensimmäistä):\n{y_val[:10]}\n")

y_train (10 ensimmäistä):
[2.523   1.469   2.909   1.413   5.00001 0.72    2.524   1.625   5.00001
 2.268  ]

y_test (10 ensimmäistä):
[3.979 2.279 1.721 1.865 1.489 0.656 3.446 2.356 2.33  1.921]

y_val (10 ensimmäistä):
[2.289 0.719 2.722 3.953 3.167 1.775 5.    2.675 1.051 3.397]



## Baseline mallin rakennus ja koulutus

In [20]:
# define the model architecture
model = Sequential([
    Input(shape=(8,)),
    Dense(64, activation='relu'),
    Dense(64, activation='relu'),
    Dense(64, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1)
    ])

# configure the learning algorithm
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

2026-10-29 12:46:28.992298: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-10-29 12:46:28.992428: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-10-29 12:46:28.992444: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1793270788.992719 5182292 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1793270788.992936 5182292 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [21]:
history = model.fit(x_train, y_train, epochs=500, batch_size=32, validation_data=(x_val, y_val), verbose=2)

Epoch 1/500


2026-10-29 12:46:42.713503: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


104/104 - 4s - 35ms/step - loss: 1.5052 - mae: 0.9242 - val_loss: 0.5582 - val_mae: 0.5474
Epoch 2/500
104/104 - 2s - 15ms/step - loss: 0.6307 - mae: 0.5813 - val_loss: 0.5514 - val_mae: 0.5298
Epoch 3/500
104/104 - 2s - 15ms/step - loss: 0.6093 - mae: 0.5714 - val_loss: 0.6189 - val_mae: 0.5833
Epoch 4/500
104/104 - 2s - 16ms/step - loss: 0.6428 - mae: 0.5881 - val_loss: 0.5203 - val_mae: 0.5176
Epoch 5/500
104/104 - 2s - 16ms/step - loss: 0.6374 - mae: 0.5833 - val_loss: 0.4972 - val_mae: 0.5031
Epoch 6/500
104/104 - 2s - 16ms/step - loss: 0.9064 - mae: 0.6779 - val_loss: 0.5707 - val_mae: 0.5467
Epoch 7/500
104/104 - 2s - 16ms/step - loss: 0.6188 - mae: 0.5729 - val_loss: 1.6398 - val_mae: 1.0712
Epoch 8/500
104/104 - 2s - 16ms/step - loss: 0.8563 - mae: 0.6849 - val_loss: 0.6674 - val_mae: 0.5978
Epoch 9/500
104/104 - 2s - 16ms/step - loss: 0.9492 - mae: 0.7078 - val_loss: 1.9754 - val_mae: 1.0243
Epoch 10/500
104/104 - 2s - 16ms/step - loss: 1.3146 - mae: 0.8378 - val_loss: 0.5383

KeyboardInterrupt: 